#### **Introduction to Numerical Integration** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This is a notebook to show why molecular dynamics simulations use symplectic integrators like Verlet, Velocity-Verlet, or Leapfrog
instead of Euler integration. It is designed not only to introduce the reader to some math, but also some simple coding so that they can get their toes wet.

##### **The Euler algorithm**


In [ ]:
# --- import our libraries ---
import numpy as np
import pandas as pd

We will use the simple simulation of a ball on a spring (harmonic oscillator). For this, we need certain constants.
Feel free to change these.

In [ ]:
# --- Parameters of our spring ---
k = 50 # spring constant
m = 1 # mass of the ball on the spring
xo = 0 # equilibrium position of the spring; different than x0 lol
# derived spring values
w = np.sqrt(k/m)

We will first explore how Euler integration works. The basic principle of this kind of numerical integration is to approximate the 
integration function using a Taylor expansion:
$$ x(t+dt) = x(t) + \frac{\partial x(t)}{\partial t}dt + \frac{\partial x(t)^2}{\partial t^2}\frac{dt^2}{2!} + \frac{\partial x(t)^3}{\partial t^3}\frac{dt^3}{3!} + \frac{\partial x(t)^4}{\partial t^4}\frac{dt^4}{4!} + ... $$

For Euler integration, we include only the first derivative, and exclude any higher order terms. As in:
$$ x(t+dt) = x(t) + \frac{\partial x(t)}{\partial t}dt + \mathcal{O}(dt^2) $$

where $\mathcal{O}(dt^2)$ indicates truncation error that scales with $dt^2$. For dynamics, we know that the derivate of position
with respect to time is velocity. Thus, Euler integration proposes to propogate the dynamics of our system as:

$$ x(t+dt) = x(t) + v(t)dt $$

So the position of a body at a step $dt$ in the future is given by the current position $x(t)$ and a constant velocity $v(t)$ 
over that time. As a simple example: you are walking in the positive $x$ direction at 3 mph and are at $x=1$ miles. In
one hour you will be at $x=4$. *The error from Euler integration essentially comes from assuming that velocity is constant over 
a timestep*, as we have truncated terms relating to acceleration ($dt^2$). However, we know that velocity is not constant, and so 
we must also update velocities, which we will do that same way:

$$ v(t+dt) = v(t) + \frac{\partial v}{\partial t}dt + \mathcal{O}(dt^2) $$

or similarly, 

$$ \frac{\partial x(t+dt)}{\partial t} = \frac{\partial x(t)}{\partial t} + \frac{\partial x(t)^2}{\partial t^2}{dt} + \mathcal{O}(dt^2)$$

where the derivative of velocity with respect to time is the acceleration.

Thus, Euler integration requires the following two equations: 
$$ x(t+dt) = x(t) + v(t)dt $$
$$ v(t+dt) = v(t) + a(t)dt $$

If at every time point we can calculate the instantaneous acceleration (e.g. from Newton's Laws), we can propogate the position of the system.

One question that arises is: is Euler integration time-reversible? That is, if we start our simulation at some $x(t)$ and $v(t)$, propogate $n$ number of $dt$ steps to $x(t+ndt)$ and $v(t+ndt)$, can we also start a simulation at  $x(t+ndt)$ and $v(t+ndt)$ and propogate reverse time ($-dt$) to get back our starting conditions $x(t)$ and $v(t)$?

In [ ]:
def euler(x0,v0,dt,w,num_osc): 
    """
    Simulates a ball on a spring using Euler integration. 
    Args:
        x0      (float): Initial position of ball.
        v0      (float): Initial velocity of ball.
        dt      (float): Integration timestep.
        w       (float): Derived spring value relating the spring stiffness k and mass of ball m.
        num_osc (float): The total number of osccillations to simulate.

    Returns:
        (pandas.DataFrame) containing the time, position, and velocity of the ball on a spring.
    """
    # --- Initialization ---
    freq = w/(2*np.pi)              # frequency of our spring
    length = int(num_osc/(freq*dt)) # how long the simulation will run for 
    x = np.zeros(length)            # initialize our position vector as 0s
    v = np.zeros(length)            # initialize our velocity vector as 0s
    t = np.zeros(length)            # initialize our time vector as 0s
    x[0] = x0 # starting x condition
    v[0] = v0 # starting velocity condition

    # --- Simulation loop (Euler integration) --- 
    for i in range(length-1):
        x[i+1] = x[i] + v[i]*dt # update position from first order Taylor Expansion
        v[i+1] = v[i] - w**2*(x[i] - xo)*dt  # update velocity from first order Taylor Expansion
        t[i+1] = t[i] + dt # march through time
    df = pd.DataFrame({'time': t, 'position': x, 'velocity': v}, columns=['time', 'position', 'velocity']) # output as a dataframe
    return df # dataframes make analyzing data a breeze


In [ ]:
sim1 = euler(2,1,0.00001,w,10) # run a simulation 
display(sim1) # print out the dataframe for you to visually inspect

In [ ]:
# --- set standard plotting conditions, like plot size etc.  ---
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(rc={"figure.dpi":300, 'savefig.dpi':300}, style='white')
plt.figure(figsize=(1,1))

def plot_position_velocity(sim):
    """"
    a function that takes the DataFrame returned by our simulation routine
    and plots both the position and velocity on the same graph.
    Args:
        sim (pandas DataFrame): returned by our simulation routine. 
                                expected columns: ['time', 'position', 'velocity'] 
    returns plt.show() to immediately show the graph.
    """
    # --- generate the figure and first sub-axis to plot position ---
    fig, ax1 = plt.subplots()   # make the objects
    color1 = 'tab:orange'       # whatever color you'd like
    ax1.set_xlabel('simulation time')
    ax1.set_ylabel('position', color=color1)
    ax1.plot(sim.time, sim.position, color=color1)
    ax1.tick_params(axis='y', labelcolor=color1)

    # --- generate the second sub-axis to plot velocity
    ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis
    color2 = 'tab:blue'
    ax2.set_ylabel('velocity', color=color2)  # we already handled the x-label with ax1
    ax2.plot(sim.time, sim.velocity, color=color2)
    ax2.tick_params(axis='y', labelcolor=color2)

    # --- sizing of our figure ---
    fig.set_figwidth(6) 
    fig.set_figheight(2)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    return plt.show()

In [ ]:
plot_position_velocity(sim1)
# when do we expect to see maximal (unisgned) velocities? 
# Does our graph show that our simulation produces that?

Now let's explore what happens as we increase the timestep of our simulations while keeping the number of oscillations the same.

In [ ]:
# bigger time step
sim2 = euler(2,1,0.0001,w,10)
plot_position_velocity(sim2)

Did we lose any accuracy of our simulation? Let's keep pushing our timestep.

In [ ]:
sim3 = euler(2,1,0.001,w,10)
plot_position_velocity(sim3) # what's happening to the peaks of the graph?

In [ ]:
sim4 = euler(2,1,0.01,w,10)
plot_position_velocity(sim4)

Oh no! What happened? Why does our system start blowing up? Let's revisit the idea of time reversibility. Let's consider a single timestep:

$$ x(t+dt) = x(t) + v(t)dt $$
$$ v(t+dt) = v(t) + a(t)dt $$

Now let's assume we start at time $t+dt$ and propogate backwards. We don't know what this new step will equal, so we'll call it as just $x$:

$$ x = x(t+dt) + v(t+dt)(-dt) $$

Solve for $x(t+dt)$:

$$ x(t+dt) = x + v(t+dt)dt $$

We now see that the Euler integration is time-irreversible, because if we substitue  $x(t)$ for $x$ we do *not* recover our starting equation. They differ as the Euler integration scheme uses $v(t)$ to propogate, whereas this one would use $v(t+dt)$. In other words, because
each timestep uses the instaneous velocity in order to propgate the equations of motion (and not, say, the average velocity predicted between two time points), our algorithm is problematic. In practice, this is related to the fact that the *energy between two timesteps is not conserved.* We can see this by plotting the energy of our system at each time point.

In [ ]:
# calculate the total energy of the spring in each simulation
E_sim1 = k/2*(sim1.position)**2 + m/2*(sim1.velocity)**2
E_sim2 = k/2*(sim2.position)**2 + m/2*(sim2.velocity)**2
E_sim3 = k/2*(sim3.position)**2 + m/2*(sim3.velocity)**2
E_sim4 = k/2*(sim4.position)**2 + m/2*(sim4.velocity)**2

In [ ]:
# I didn't bother to make this a function
# if you want Python practice, you can do it yourslef :) 

fig, ax1 = plt.subplots()
ax1.plot(sim1.time, E_sim1, label='E_sim1')
ax1.plot(sim2.time, E_sim2, label='E_sim2')
ax1.plot(sim3.time, E_sim3, label='E_sim3')
#ax1.plot(sim4.time, E_sim4, label='E_sim4') #comment out this line to easily compare sims 1-3.
ax1.set_xlabel('simulation time')
ax1.set_ylabel('energy')
ax1.legend()
fig.tight_layout()
fig.set_figwidth(6) 
fig.set_figheight(2)
plt.show()


### **The Velocity Verlet algorithm**
With the Veloicty Verlet algorithm, we will used a second order Taylor expansion. That is:

$$ x(t+dt) = x(t) + \frac{\partial x(t)}{\partial t}dt +\frac{1}{2}\frac{\partial x(t)^2}{\partial t^2}dt^2  $$

We know that the second derivative of position with respect to time is acceleration: we will immediately rewrite acceleration in terms of force such that it is a quantity that is directly calculatable.

$$ x(t+dt) = x(t) + v(t)dt +\frac{1}{2m}F(t)dt^2  $$

The nice trick employed in this algorithm is that for velocity we will update the new velocity based upon the *average force* between two timesteps. That is:

$$ v(t+dt) = v(t) + \frac{1}{2m}\big[F(t) + F(t+dt)\big]dt $$

In other words, we are trying to account for the fact that acceleration is not constant during a timestep and will assume that our timestep is sufficiently small such that the average acceleration is a good approximation (i.e. that acceleration changes linearly between the two time points).

Let's use a Velocity Verlet integration scheme and see how it handles our harmonic oscillator.


In [ ]:
def velocity_verlet(x0,v0,dt,w,num_osc):
    """
    A function that simulates a ball on a spring using Velocity Verlet integration.
    Args:
        x0      (float): Initial position of ball.
        v0      (float): Initial velocity of ball.
        dt      (float): Integration timestep.
        w       (float): Derived spring value relating the spring stiffness k and mass of ball m.
        num_osc (float): The total number of osccillations to simulate.
    Returns:
        (pandas.DataFrame) containing the time, position, and velocity of the ball on a spring.
    """
    # --- Initialize ---
    freq = w/(2*np.pi)              # frequency of our spring
    length = int(num_osc/(freq*dt)) # how long the simulation will run for 
    x = np.zeros(length)            # initialize our position vector as 0s
    v = np.zeros(length)            # initialize our velocity vector as 0s
    t = np.zeros(length)            # initialize our time vector as 0s
    x[0] = x0
    v[0] = v0

    # --- Velocity Verlet integration ---
    for i in range(length-1): 
        x[i+1] = x[i] + v[i]*dt - (w**2*(x[i] - xo)*dt**2)/2
        v[i+1] = v[i] - w**2*( (x[i] - xo) + (x[i+1]-xo) )*dt/2 # notice the signage! Force is negative derivative of energy
        t[i+1] = t[i] + dt
    df = pd.DataFrame({'time': t, 'position': x, 'velocity': v}, columns=['time', 'position', 'velocity']) # output as a dataframe
    return df # dataframes make analyzing data a breeze

In [ ]:
vv_sim1 = velocity_verlet(2,1,0.00001,w,10) # run a simulation 
vv_sim1 # print out the dataframe for you to visually inspect
plot_position_velocity(vv_sim1)

In [ ]:
vv_sim2 = velocity_verlet(2,1,0.0001,w,10) # run a simulation 
plot_position_velocity(vv_sim2)

In [ ]:
vv_sim3 = velocity_verlet(2,1,0.001,w,10) # run a simulation 
plot_position_velocity(vv_sim3)

In [ ]:
vv_sim4 = velocity_verlet(2,1,0.01,w,10) # run a simulation 
plot_position_velocity(vv_sim4)

In [ ]:
# calculate the total energy of the spring in each simulation
E_vv_sim1 = k/2*(vv_sim1.position)**2 + m/2*(vv_sim1.velocity)**2
E_vv_sim2 = k/2*(vv_sim2.position)**2 + m/2*(vv_sim2.velocity)**2
E_vv_sim3 = k/2*(vv_sim3.position)**2 + m/2*(vv_sim3.velocity)**2
E_vv_sim4 = k/2*(vv_sim4.position)**2 + m/2*(vv_sim4.velocity)**2

In [ ]:
# Let's compare energy drift from Euler and Velocity Verlet
# again, if the reader wants to practice their Python they could make this a function :) 
fig, ax1 = plt.subplots()
ax1.plot(sim1.time, E_sim1, label='Euler sim1')
ax1.plot(vv_sim1.time, E_vv_sim1, label='VV sim1')
ax1.set_xlabel('simulation time')
ax1.set_ylabel('energy')
ax1.legend()
fig.set_figwidth(6) 
fig.set_figheight(2)
fig.tight_layout()
plt.show()

In [ ]:
# Let's compare energy drift from Euler and Velocity Verlet
fig, ax1 = plt.subplots()
ax1.plot(sim2.time, E_sim2, label='Euler sim2')
ax1.plot(vv_sim2.time, E_vv_sim2, label='VV sim2')
ax1.set_xlabel('simulation time')
ax1.set_ylabel('energy')
ax1.legend()
fig.set_figwidth(6) 
fig.set_figheight(2)
fig.tight_layout()
plt.show()

In [ ]:
# Let's compare energy drift from Euler and Velocity Verlet
fig, ax1 = plt.subplots()
ax1.plot(sim3.time, E_sim3, label='Euler sim3')
ax1.plot(vv_sim3.time, E_vv_sim3, label='VV sim3')
ax1.set_xlabel('simulation time')
ax1.set_ylabel('energy')
ax1.legend()
fig.set_figwidth(6) 
fig.set_figheight(2)
fig.tight_layout()
plt.show()

In [ ]:
# Let's compare energy drift from Euler and Velocity Verlet
fig, ax1 = plt.subplots()
ax1.plot(sim4.time, E_sim4, label='Euler sim4')
ax1.plot(vv_sim4.time, E_vv_sim4, label='VV sim4')
ax1.set_xlabel('simulation time')
ax1.set_ylabel('energy')
ax1.legend()
fig.set_figwidth(6) 
fig.set_figheight(2)
fig.tight_layout()
plt.show()

### **Homework for the reader:**

1. Is Velocity Verlet time-reversible? Prove this.

2. Can you write up a function that implements the Verlet or Leap-Frog Verlet integration?

3. Plot a density distribution of the ensemble of states sampled like we did in the Monte Carlo/Importance Sampling worksheet. What do you notice comparing the Euler and VV distributions? What do you notice about how using different timesteps in the VV routine affects the distribution? 

4. Why do we not use Runge-Kutta or Predictor Corrector type algorithms?

5. Do we care about energy conservation in MD simulations at constant temperature (not at constant energy anyway)? Why or why not?

6. In addition to caring for energy drift better than Euler or other types of integrators, Verlet-type integrators also boost performance because we can simulate with larger timesteps. Do we have to do any additional calculations for Verlet-type integrators that could offset the boost in performance?
